# Algoritmos de optimización - Seminario<br>
Nombre y Apellidos: Juan Carlos Vereda Ruiz y Rigel Chulia Ortega  <br>
Url: https://github.com/RigelChuliaO/Algoritmos-de-Optimizaci-n/blob/f772569821b83976e6fb6f4f284c5fb48439d0bc/Seminario_Algoritmo_JC_Vereda_R_Chulia_V2.ipynb<br>
Problema:
> 1. Sesiones de doblaje <br>
>2. Organizar los horarios de partidos de La Liga<br>
>3. Combinar cifras y operaciones

Descripción del problema:(copiar enunciado)

Problema 3. Combinar cifras y operaciones
• El problema consiste en analizar el siguiente problema y diseñar un algoritmo que lo resuelva.
• Disponemos de las 9 cifras del 1 al 9 (excluimos el cero) y de los 4 signos básicos de las
operaciones fundamentales: suma(+), resta(-), multiplicación(*) y división(/)
• Debemos combinarlos alternativamente sin repetir ninguno de ellos para obtener una
cantidad dada. Un ejemplo sería para obtener el 4:<br>
4+2-6/3x1 = 4

 Debe analizarse el problema para encontrar todos los valores enteros posibles planteando las
siguientes cuestiones:- ¿Qué valor máximo y mínimo se pueden obtener según las condiciones del problema?- ¿Es posible encontrar todos los valores enteros posibles entre dicho mínimo y máximo ?
• Nota: Es posible usar la función de python “eval” para evaluar una expresión
                                        

In [1]:
# Librerias utilizadas

from itertools import permutations
import random
import time
from fractions import Fraction

In [5]:
# Funcion Auxiliar para el calculo de las espresiones.
#Evaluación exacta con precedencia usando Fraction

#Esta función evalúa d1 op1 d2 op2 d3 op3 d4 op4 d5 sin usar eval

OPS = ['+', '-', '*', '/']

def eval_nueva(digits, ops):
    """
    Evalúa con precedencia estándar usando el truco:
      total acumula sumas/restas de términos,
      term acumula el término actual con * y /.
    Todo con Fraction para exactitud.
    """
    # Inicializamos
    term = Fraction(digits[0], 1)
    total = Fraction(0, 1)

    for i, op in enumerate(ops):
        x = Fraction(digits[i+1], 1)

        if op == '*':
            term *= x
        elif op == '/':
            term /= x  # seguro: x != 0
        elif op == '+':
            total += term
            term = x
        elif op == '-':
            total += term
            term = -x
        else:
            raise ValueError(f"Operador no soportado: {op}")

    return total + term

# Ejemplo del enunciado: 2*3+1/4-8 = -5.75
print(eval_nueva([2,3,1,4,8], ['*','+','/','-']))

-7/4


# (*)¿Cuantas posibilidades hay sin tener en cuenta las restricciones?<br>








Primero definimos las restricciones del problema:
1. **Restricción de Dígitos**

    •	Solo cifras del 1 al 9: Se excluye explícitamente el número 0.
    •	Sin repetición cifras: solo se puede usar 5 cifras de las posible y no se pueden repetir.
2. **Restricción de Operadores**

    •	Operadores básicos: Solo se pueden usar los cuatro signos fundamentales: suma (+), resta (-), multiplicación (*) y división (/).
    •	Sin repetición de operadores: Cada uno de los 4 operadores debe aparecer exactamente una vez en la expresión.
3. **Restricción de Estructura**

    •	Combinación alternativa: Los elementos deben colocarse intercalados. Esto obliga a que la estructura sea siempre: Número → Operador → Número → Operador → Número → Operador → Número → Operador → Número
4. **Restricción de Resultados**

    • Valores enteros: Aunque el cálculo intermedio (especialmente la división) pueda generar decimales, el enunciado pregunta por los valores enteros posibles.


Calculamos las posibilidades sin esas restriciones:
1. Posiciones de los números: Disponemos de 9 dígitos (1-9) para 5 huecos. Al permitir la repetición, cada hueco tiene 9 opciones posibles:$$9 \times 9 \times 9 \times 9 \times 9 = 9^5 = 59,049 \text{ combinaciones de números}$$ Posiciones de los operadores: Disponemos de 4 operadores ($+, -, \times, \div$) para 4 huecos. Si permitimos que se repitan (por ejemplo, usar siempre la suma), cada hueco tiene 4 opciones:$$4 \times 4 \times 4 \times 4 = 4^4 = 256 \text{ combinaciones de operadores}$$ Total de casos posibles: El total de expresiones distintas que el algoritmo tendría que evaluar se obtiene multiplicando ambos resultados:$$\textbf{Total} = 59,049 \times 256 = \mathbf{15,116,544} \text{}$$

# ¿Cuantas posibilidades hay teniendo en cuenta todas las restricciones.

Ahora apalicando las restricciones nos quedaria:

Calculamos las permutaciones de 9 elementos tomados de 5 en 5:$$P(9, 5) = \frac{9!}{(9-5)!} = 9 \times 8 \times 7 \times 6 \times 5 = 15,120$$Posiciones de los operadoresTenemos 4 operadores ($+, -, \times, \div$) para 4 huecos. Esto es una permutación simple de 4:$$P_4 = 4! = 4 \times 3 \times 2 \times 1 = 24$$Total de casos posiblesEl total de combinaciones se obtiene mediante el producto de ambas posibilidades:$$15,120 \times 24 = 362,880 \text{ combinaciones}$$

# Modelo para el espacio de soluciones<br>
(*) ¿Cual es la estructura de datos que mejor se adapta al problema? Argumentalo.(Es posible que hayas elegido una al principio y veas la necesidad de cambiar, arguentalo)


La estructura de datos que mejor se adapta al problema depende de dos necesidades:
1. Representar de forma ordenada una expresión candidata
2. Almacenar los resultados obtenidos evitando duplicados.

Para representar cada operación, lo más adecuado es usar **tuplas o listas**: una tupla/lista de 5 dígitos (d1,d2,d3,d4,d5) y una tupla/lista de 4 operadores (op1,op2,op3,op4). Esto es apropiado porque el orden importa y además itertools.permutations devuelve precisamente tuplas, lo que simplifica la generación de combinaciones, que utilizaremos para el caso de la fuerza bruta.

Para almacenar los resultados enteros distintos se utiliza un **conjunto (set)**, ya que elimina duplicados automáticamente y permite comprobar pertenencia de forma eficiente. Esto facilita el análisis posterior (obtener mínimo, máximo y comprobar si el intervalo [min,max] está completo).

Para evaluar con exactitud (especialmente por las divisiones) emplearemos el tipo **Fraction**, que evita errores de redondeo y permite comprobar si un resultado es entero verificando que su denominador sea 1.

Opcionalmente, si podria utilizar un **diccionario (dict)** para guardar cada operacion valida con su resultado.

# Según el modelo para el espacio de soluciones<br>
(*)¿Cual es la función objetivo?




Dado un conjunto de cifras $D = \{1, 2, \dots, 9\}$ y un conjunto de operadores $O = \{+, -, *, /\}$, la función objetivo $f$ se define sobre una secuencia combinada $S = (d_1, o_1, d_2, o_2, d_3, o_3, d_4, o_4, d_5)$ tal que:$$f(S) = (((d_1 \ o_1 \ d_2) \ o_2 \ d_3) \ o_3 \ d_4) \ o_4 \ d_5$$

Donde:Cada $d_i$ es una cifra única de $D$ (sin repetir).

Cada $o_i$ es un operador único de $O$ (sin repetir).

La evaluación de la función $f(S)$ no se realiza de manera puramente secuencial, sino que sigue la jerarquía de operaciones aritmética estándar (prioridad de multiplicación y división sobre suma y resta). Técnicamente, el algoritmo descompone la secuencia en "términos" que se consolidan en un "total" acumulado solo cuando se encuentra un operador de menor jerarquía ($+$ o $-$), garantizando así la precisión matemática del resultado.


# (*)¿Es un problema de maximización o minimización?



No se trata de un problema de optimización clásico, ya que no se busca maximizar ni minimizar un valor concreto. El objetivo es explorar todo el espacio de soluciones para encontrar todos los resultados enteros posibles.

Por tanto, puede considerarse un problema de enumeración o exploración del espacio de soluciones más que un problema de maximización o minimización.

# Diseña un algoritmo para resolver el problema por fuerza bruta

In [2]:
# Conjunto de Datos del enunciado
OPS = ['+', '-', '*', '/']
DIGITOS =[1,2,3,4,5,6,7,8,9]



In [3]:

def Fuerza_Bruta(digitos_input, ops_input, guardar_validas=True):
    # Definimos cuántas cifras necesitamos según los operadores (N_ops + 1)
    n_ops = len(ops_input) 
    n_cifras_necesarias = n_ops + 1
    
    # Permutaciones de operadores (4! = 24)
    op_perms = list(permutations(ops_input, n_ops))

    S = set() 
    operaciones_validas = {}  
    nodos_visitados = 0
    tiempo_inicial = time.perf_counter() 

    # IMPORTANTE: Permutaciones de los dígitos de entrada tomados de N en N
    # Si entran 9 dígitos y necesitamos 5, hará P(9,5) = 15,120
    for digs in permutations(digitos_input, n_cifras_necesarias):
        for ops in op_perms:
            val = eval_nueva(digs, ops)
            nodos_visitados += 1

            if val.denominator == 1: 
                iv = int(val)
                S.add(iv)
                if guardar_validas and iv not in operaciones_validas:
                    expr = str(digs[0])
                    for i in range(len(ops)):
                        expr += f"{ops[i]}{digs[i+1]}"
                    operaciones_validas[iv] = expr
                            
    tiempo_final = time.perf_counter()
    #return S, operaciones_validas, nodos_visitados, (tiempo_final - tiempo_inicial)
    return S,nodos_visitados, (tiempo_final - tiempo_inicial)



In [6]:

#Ejecucion de la Fuerza Bruta para resolver el problema con las restricciones definidas.

S_brute, n_expr, t_brute = Fuerza_Bruta(DIGITOS,OPS,guardar_validas=True)

print("DIGITOS",DIGITOS)
print("OPERADORES",OPS)
print("EXPRESIONES EVALUADAS:", n_expr)
print("TIEMPO (s):", round(t_brute, 4))
print("ENTEROS DISTINTOS |S|:", len(S_brute))

# Para el caso de que el resultado de nodos visitados sea 0 no de error 
if len(S_brute) > 0:
    minimo = min(S_brute)
    maximo = max(S_brute)
else:
    #Forzamos el valro del MIN y MAX a "-" por que no hay
    minimo = "-"
    maximo = "-"
print("MIN:", minimo, "MAX:", maximo)

DIGITOS [1, 2, 3, 4, 5, 6, 7, 8, 9]
OPERADORES ['+', '-', '*', '/']
EXPRESIONES EVALUADAS: 362880
TIEMPO (s): 2.6273
ENTEROS DISTINTOS |S|: 147
MIN: -69 MAX: 77


### Calcula la complejidad del algoritmo por fuerza bruta

En el caso de tu algoritmo de cifras y operadores (donde $n=9$, $m=4$ y $k=5$):

Como $n$ y $m$ son constantes (valores fijos de 9 y 4), la complejidad Big O técnica para esa ejecución específica es $O(1)$, ya que el número de pasos no crece con la entrada (siempre son 362,880).

Sin embargo, para $n$ dígitos y $m$ operadores, la complejidad sería:$$\mathbf{O(n^k \times m!)}$$(Donde $k$ es el número de posiciones a llenar).

Aunque el algoritmo realiza operaciones de orden polinómico $O(n^k)$ para la validación, la complejidad asintótica del algoritmo de fuerza bruta está determinada por la generación del espacio de búsqueda, que es de orden factorial $O(m!)$. Por tanto, la asíntota dominante es $O(m!)$"

$$\mathbf{O(m!)}$$

# (*)Diseña un algoritmo que mejore la complejidad del algortimo por fuerza bruta. Argumenta porque crees que mejora el algoritmo por fuerza bruta

In [62]:
from functools import lru_cache

def backtracking_memoizado(digitos_input, ops_input):
    tiempo_inicial = time.perf_counter()
    n_cifras_necesarias = len(ops_input) + 1 # para que se adapte a los datos de entrada
    
    nodos_visitados = [0] # Lista para cont

    @lru_cache(maxsize=None)
    def dfs(mask_digits, mask_ops, pos, total, term):

        if pos == n_cifras_necesarias:
            val = total + term
            nodos_visitados[0] += 1  # Para contar igual que Fuerza Bruta, las expresiones totales realizadas
            if val.denominator == 1:
                return frozenset([int(val)])
            return frozenset()

        resultados = set() # Aqui almacenamos los valores enteros encontrados
        # Control de operadores usados
        for i, op in enumerate(ops_input):
            if mask_ops & (1 << i):
                continue
            # Control de digitos
            for d in digitos_input:
                bit_d = 1 << (d - 1)

                if mask_digits & bit_d:
                    continue

                x = Fraction(d, 1)
                new_total = total
                new_term = term

                if op == '*':  # Realiza la operacion
                    new_term = term * x
                elif op == '/':
                    new_term = term / x
                elif op == '+':
                    new_total = total + term
                    new_term = x
                elif op == '-':
                    new_total = total + term
                    new_term = -x
                    
                # RECURSIVIDAD
                resultados.update(
                    dfs(
                        mask_digits | bit_d,
                        mask_ops | (1 << i),
                        pos + 1,
                        new_total,
                        new_term
                    )
                )
        return frozenset(resultados) # Eliminamos
    S = set()
    
    for d in digitos_input:
        mask_digits = 1 << (d - 1)  # Creamos la mascara para saber lo que esta usado
        S.update(
            dfs(
                mask_digits,
                0,
                1,
                Fraction(0, 1),
                Fraction(d, 1)
            )
        )

    tiempo_final = time.perf_counter()

    return S, nodos_visitados[0], tiempo_final - tiempo_inicial

In [63]:

#Ejecucion de la solucion de MEMOIZACIÓN
OPS = ('+', '-', '*', '/')
DIGITOS =[1,2,3,4,5,6,7,8,9]

S_memo, nodos, tiempo_memo = backtracking_memoizado(DIGITOS,OPS)

print("DIGITOS",DIGITOS)
print("OPERADORES",OPS)
print("EXPRESIONES EVALUADAS:", nodos)
print("TIEMPO (s):", round(tiempo_memo, 4))
print("ENTEROS DISTINTOS |S|:", len(S_memo))

# Para el caso de que el resultado de nodos visitados sea 0 no de error 
if len(S_memo) > 0:
    minimo = min(S_memo)
    maximo = max(S_memo)
else:
    #Forzamos el valro del MIN y MAX a "-" por que no hay
    minimo = "-"
    maximo = "-"
print("MIN:", minimo, "MAX:", maximo)

DIGITOS [1, 2, 3, 4, 5, 6, 7, 8, 9]
OPERADORES ('+', '-', '*', '/')
EXPRESIONES EVALUADAS: 76670
TIEMPO (s): 0.9146
ENTEROS DISTINTOS |S|: 147
MIN: -69 MAX: 77


La diferencia fundamental entre ambos algoritmos no radica en el resultado, sino en el esfuerzo realizado para alcanzarlo, ya que el objetivo del problema es encontrar el resultado final correcto y no una aproximación, esto incluye saber el numero de soluciones enteras distintas, su min y su max 

Para los datos del problema definimos dos conjuntos de datos
OPS = ['+', '-', '*', '/']
DIGITOS =[1,2,3,4,5,6,7,8,9]

El resultado final es Min:-69,
Max: 77
Soluciones enteras distintas: 147.


•	Fuerza Bruta: Este algoritmo genera y evalúa la totalidad del espacio de soluciones sin excepción, lo que supone un total de 362,880 expresiones evaluadas.

•	Memoizado: Gracias a la implementación de lru_cache (memoización), el algoritmo optimiza el árbol de búsqueda visitando solo 76670 nodos. Esto significa que el algoritmo evita procesar 286,210 combinaciones (un 78.87% menos) al identificar estados (subproblemas) que ya habían sido calculados previamente, aprovechando los resultados almacenados en lugar de recalcularlos.

Explicacion de la Memoización:
A diferencia de la fuerza bruta que repite cálculos, la memoización utiliza un almacenamiento intermedio para guardar los resultados de las llamadas recursivas:.

                    @lru_cache(maxsize=None)
                    def dfs(mask_digits, mask_ops, pos, total, term):
                    # ... lógica de recursión ...

Este paso es vital, ya que transforma la eficiencia del algoritmo:

Detección de estados repetidos: El algoritmo identifica cuando llega a una combinación de mask_digits, mask_ops, pos, total y term que ya ha procesado anteriormente.

Recuperación instantánea: En lugar de desplegar toda la rama recursiva de nuevo, el sistema recupera el resultado directamente de la caché, "atajando" el proceso.

Reducción de redundancia: Mientras que la Fuerza Bruta debe procesar cada expresión de 5 dígitos desde el inicio independientemente de si ha visto esa combinación antes, el algoritmo memoizado aprovecha los cálculos parciales de los niveles superiores del árbol.

La operación de gestionar la caché tiene un coste de gestión en memoria, pero el beneficio de evitar cientos de miles de pasos recursivos supera con creces el pequeño coste de realizar las consultas a la caché en cada nodo, como podemos observar en los resultados obtenidos.

## Comparativa Fuerza Bruta Vs Memoizadó


En este apartado, se detalla la implementación de una función diseñada para evaluar el rendimiento y la precisión de los dos enfoques desarrollados: Fuerza Bruta y Backtracking Memoizado. Esta herramienta permite someter a ambos algoritmos a diferentes juegos de datos (dígitos y operadores) para medir su eficiencia bajo las mismas condiciones.

**Funcionamiento de la Comparativa**<br>
La función de comparación recibe como parámetros los conjuntos de dígitos y operadores seleccionados. Una vez ejecutados ambos procesos, genera de forma automática una tabla que permite visualizar el impacto de la optimización en los siguientes indicadores clave:<br>

**Nodos Visitados:** Representa el número total de operaciones completas realizadas. Es el indicador principal de la complejidad computacional de cada algoritmo y permite visualizar cuánto esfuerzo se ahorra mediante la reutilización de estados (memoización).<br>

**Tiempo de Ejecución:** Mide los segundos (s) requeridos para completar la búsqueda. Cabe destacar que en conjuntos de datos pequeños, este valor es altamente sensible al hardware y a los procesos en segundo plano del sistema, por lo que su relevancia aumenta a medida que el volumen de datos crece.<br>

**Métricas de Resultados** [MIN, MAX, N]: Para validar la fiabilidad de los algoritmos, se comparan los valores de las soluciones el valor mínimo, el valor máximo y el número total de enteros distintos generados. El hecho de que ambos algoritmos arrojen resultados idénticos confirma la exactitud de ambos enfoques.


In [65]:
def comparar_resultados(digitos_entrada, operadores_entrada):

    # Ejecutamos la Fuerza Bruta
    S_fb,nodos_fb, t_fb = Fuerza_Bruta(digitos_entrada, operadores_entrada, guardar_validas=False)
    
    # Ejecutamos la Memoizacion
    S_bt, nodos_bt, t_bt = backtracking_memoizado(digitos_entrada, operadores_entrada)

    # Cálculo de porcentajes de mejora (Reducción de esfuerzo)
    mejora_nodos = ((nodos_fb - nodos_bt) / nodos_fb * 100) if nodos_fb > 0 else 0
    mejora_tiempo = ((t_fb - t_bt) / t_fb * 100) if t_fb > 0 else 0

    # Tabla de Resultados (Formato exacto de la imagen)
    
    
    min_fb = min(S_fb) if S_fb else "-"
    max_fb = max(S_fb) if S_fb else "-"
    min_bt = min(S_bt) if S_bt else "-"
    max_bt = max(S_bt) if S_bt else "-"
    
    res_fb_str = f"min:{min_fb} max:{max_fb} Nº enteros:{len(S_fb)}"
    res_bt_str = f"min:{min_bt} max:{max_bt} Nº enteros:{len(S_bt)}"

    header = f"{'Métrica':<25} | {'Fuerza Bruta':<30} | {'Memoización':<30} | {' Mejora (%)':<12}"
    line = "-" * len(header)
    print(header)
    print(line)
    print(f"{'Nodos Visitados':<25} | {nodos_fb:<30.0f} | {nodos_bt:<30.0f} | {mejora_nodos:>10.2f}%")
    print(f"{'Tiempo Medio (s)':<25} | {t_fb:<30.4f} | {t_bt:<30.4f} | {mejora_tiempo:>10.2f}%")
    print(f"{'Resultados [MIN, MAX, Nº]':<25} | {res_fb_str:<30} | {res_bt_str:<30} | {'N/A':>11}") 
    print(line)


# --- Ejemplo de ejecución ---
# 1 Caso inicial del problema, 9 digitos distintos y 4 operadores basicos
MIS_DIGITOS = [1, 2, 3, 4, 5, 6, 7, 8, 9]
MIS_OPERADORES = ['+', '-', '*', '/']

comparar_resultados(MIS_DIGITOS, MIS_OPERADORES)



Métrica                   | Fuerza Bruta                   | Memoización                    |  Mejora (%) 
----------------------------------------------------------------------------------------------------------
Nodos Visitados           | 362880                         | 76670                          |      78.87%
Tiempo Medio (s)          | 2.6292                         | 0.7428                         |      71.75%
Resultados [MIN, MAX, Nº] | min:-69 max:77 Nº enteros:147  | min:-69 max:77 Nº enteros:147  |         N/A
----------------------------------------------------------------------------------------------------------


In [66]:
# 2 Caso inicial del problema 6 digitos distintos y 4 operadores
MIS_DIGITOS=[5, 9, 8,6,4,1]
MIS_OPERADORES= ['-', '+','*','/']

comparar_resultados(MIS_DIGITOS, MIS_OPERADORES)

Métrica                   | Fuerza Bruta                   | Memoización                    |  Mejora (%) 
----------------------------------------------------------------------------------------------------------
Nodos Visitados           | 17280                          | 3238                           |      81.26%
Tiempo Medio (s)          | 0.1278                         | 0.0420                         |      67.11%
Resultados [MIN, MAX, Nº] | min:-63 max:74 Nº enteros:89   | min:-63 max:74 Nº enteros:89   |         N/A
----------------------------------------------------------------------------------------------------------


Podemos ver que los resultados de lo algoritmos son iguales siempre pero los nodos y tiempo se reducen.

In [67]:
# Caso sin Division
MIS_DIGITOS=[5, 9, 8,6,4]
MIS_OPERADORES= ['-', '+','*']

comparar_resultados(MIS_DIGITOS, MIS_OPERADORES)

Métrica                   | Fuerza Bruta                   | Memoización                    |  Mejora (%) 
----------------------------------------------------------------------------------------------------------
Nodos Visitados           | 720                            | 270                            |      62.50%
Tiempo Medio (s)          | 0.0052                         | 0.0033                         |      36.18%
Resultados [MIN, MAX, Nº] | min:-63 max:74 Nº enteros:64   | min:-63 max:74 Nº enteros:64   |         N/A
----------------------------------------------------------------------------------------------------------


# Según el problema (y tenga sentido), diseña un juego de datos de entrada aleatorios

Ahora que tenemos una funcion para comparar los algoritmos , vamos a crear una funcion que genere de forma aleatria los conjuntos de datos que les introduciremos.

Las condiciones para la creacion de esos conjuntos de datos son:
- No se puede poner mas de los digitos que tenga el Array de digitos de partida.
- No se puede poner menos de 2 digitos por que no se podria realizar ninguna operacion.
- Los operadores y digitos que aparezcan seran selecciondos de forma aleatoria de los arrays de partida
- Los arrays de partida seran los del enunciado de la practica:
DIGITOS = [1, 2, 3, 4, 5, 6, 7, 8, 9]
OPERADORES = ['+', '-', '*', '/']

In [51]:
import random

def generar_entrada_aleatoria(n_digitos):
    # Definición de los conjuntos de partida para la creacion de nuevos (se han tomado los del enunciado del problema)
    DIGITOS_PARTIDA = [1, 2, 3, 4, 5, 6, 7, 8, 9]
    OPERADORES_PARTIDA = ['+', '-', '*', '/']

    # Necesitamos las logitudes de los conjutnos de datos para poder controlar las restricciones a la hora de crear los nuevos.
    max_dig_posibles = len(DIGITOS_PARTIDA)
    max_ops_posibles = len(OPERADORES_PARTIDA)

    #CONDICIONES PARA LOS NUEVOS CONJUNTOS DE DATOS
    # Nunca se puede poner menos de 2 digitos para que pueda poner almenos 1 operador
    if n_digitos < 2:
        n_digitos = 2

    # No puede pedir mas digitos de los digitos que tenga el cojunto DIGITOS_PARTIDA
    if n_digitos > max_dig_posibles:
        n_digitos = max_dig_posibles # en ese caso seleciona el maximo

    # si el Nº de digitos solicita es > que el numero de operaciones, siempre sera el el numero de operaciones de 
    if n_digitos > 5:
        n_ops = 4
    else:
        n_ops = n_digitos - 1 
    
    # Generar lista de dígitos aleatorios sin repetición
    digitos_seleccionados = random.sample(DIGITOS_PARTIDA, k=n_digitos)
    operadores_seleccionados = random.sample(OPERADORES_PARTIDA, k=n_ops)
    
    return digitos_seleccionados, operadores_seleccionados # Devuelve los dos Arrays creados DIGITOS y OPS



# Aplica el algoritmo al juego de datos generado

In [52]:
# CASO 1
# Bloque de prueba
# Vamos a realizar 3 iteracciones con el maximo numero de Digitos.
X = 3
for i in range(X):
    d, o = generar_entrada_aleatoria(9)
    print(f"Ejecución {i+1}:")
    print(f"  Dígitos ({len(d)}): {d}")
    print(f"  Operadores ({len(o)}): {o}")
    comparar_resultados(d, o)

Ejecución 1:
  Dígitos (9): [5, 6, 8, 1, 2, 7, 9, 4, 3]
  Operadores (4): ['-', '/', '+', '*']
Métrica                   | Fuerza Bruta                   | Memoización                    |  Mejora (%) 
----------------------------------------------------------------------------------------------------------
Nodos Visitados           | 362880                         | 76670                          |      78.87%
Tiempo Medio (s)          | 2.6623                         | 0.7224                         |      72.87%
Resultados [MIN, MAX, Nº] | min:-69 max:77 Nº enteros:147  | min:-69 max:77 Nº enteros:147  |         N/A
----------------------------------------------------------------------------------------------------------
Ejecución 2:
  Dígitos (9): [2, 8, 1, 9, 6, 7, 5, 3, 4]
  Operadores (4): ['/', '+', '*', '-']
Métrica                   | Fuerza Bruta                   | Memoización                    |  Mejora (%) 
----------------------------------------------------------------

Salen las 3 comparaciones iguales puesto que los datos son siempre los mismos. 9 digitos distintos y 4 operadores



In [54]:
# CASO 2
# En este caso solo utilizaremos 6 digitos, 
# como tiene que seleccionarlos de los 9 de partida, el conjunto de datos de los digitos seran distintos en cada iteraccion.
# El numero de operadores sera 4
X = 3
for i in range(X):
    d, o = generar_entrada_aleatoria(6)
    print(f"Ejecución {i+1}:")
    print(f"  Dígitos ({len(d)}): {d}")
    print(f"  Operadores ({len(o)}): {o}")
    comparar_resultados(d, o)

Ejecución 1:
  Dígitos (6): [6, 9, 1, 4, 5, 2]
  Operadores (4): ['*', '/', '-', '+']
Métrica                   | Fuerza Bruta                   | Memoización                    |  Mejora (%) 
----------------------------------------------------------------------------------------------------------
Nodos Visitados           | 17280                          | 3216                           |      81.39%
Tiempo Medio (s)          | 0.1387                         | 0.0424                         |      69.42%
Resultados [MIN, MAX, Nº] | min:-51 max:57 Nº enteros:86   | min:-51 max:57 Nº enteros:86   |         N/A
----------------------------------------------------------------------------------------------------------
Ejecución 2:
  Dígitos (6): [1, 2, 9, 6, 4, 8]
  Operadores (4): ['+', '-', '/', '*']
Métrica                   | Fuerza Bruta                   | Memoización                    |  Mejora (%) 
----------------------------------------------------------------------------------

Podemos ver como el algoritmo de Memoizacion tiene una dependencia de que datos entran , no solo del numero de estos. 
La probabilidad de que las operaciones llegen al mismo resultado cambia al cambiar los digitos. 


In [59]:
#CASO 3
# en este caso forzaremos que el Array de Digitos siempre sea el mismo y solo cambiaremos los operadores
# la idea de este experimento es ver como afecta a memoizacion que cambies los operadores
DIGITOS = [5, 8, 7,3]

X = 3
for i in range(X):
    d, o = generar_entrada_aleatoria(len(DIGITOS)) # generamos los operadores y los digitos pero solo usaremos los operadores
    print(f"Ejecución {i+1}:")
    print(f"  Dígitos ({len(DIGITOS)}): {DIGITOS}")
    print(f"  Operadores ({len(o)}): {o}")
    comparar_resultados(DIGITOS, o)

Ejecución 1:
  Dígitos (4): [5, 8, 7, 3]
  Operadores (3): ['+', '*', '-']
Métrica                   | Fuerza Bruta                   | Memoización                    |  Mejora (%) 
----------------------------------------------------------------------------------------------------------
Nodos Visitados           | 144                            | 54                             |      62.50%
Tiempo Medio (s)          | 0.0011                         | 0.0011                         |       4.24%
Resultados [MIN, MAX, Nº] | min:-48 max:58 Nº enteros:18   | min:-48 max:58 Nº enteros:18   |         N/A
----------------------------------------------------------------------------------------------------------
Ejecución 2:
  Dígitos (4): [5, 8, 7, 3]
  Operadores (3): ['*', '+', '-']
Métrica                   | Fuerza Bruta                   | Memoización                    |  Mejora (%) 
--------------------------------------------------------------------------------------------------------

Podemos confirmar que al algoritmo de memoizacion le estan afectando los operadores.<br>
En estos resultados podemos ver:<br>
Valores negativos de tiempo. Por la velocidad de los procesos el tiempo no se puede tener encuenta, no es un error.<br>
Valores de min, max y Nº de enteros igual a 0. Eso es posible dado que los espacios de trabajo son pequeños y puede no tener valores enteros.Esto nos sirve tamien para confirmar que ambos algoritmos buscan la misma solucion. 


### Enumera las referencias que has utilizado(si ha sido necesario) para llevar a cabo el trabajo

Documentación General:
Python Software Foundation. (s.f.). Python 3.12.0 documentation. Recuperado el 6 de marzo de 2026, de https://docs.python.org/3/


Módulo itertools:
Python Software Foundation. (s.f.). itertools — Functions creating iterators for efficient looping. Python 3.12.0 documentation. https://docs.python.org/3/library/itertools.html


Módulo fractions:
Python Software Foundation. (s.f.). fractions — Rational numbers. Python 3.12.0 documentation. https://docs.python.org/3/library/fractions.html


Peleato, J. J. (s.f.). Backtracking. Algoritmia. https://docs.jjpeleato.com/algoritmia/backtracking


Algoritmos de Optimización:
Jose Manuel Camacho Camacho. (2026). Material y apuntes de la asignatura de Algoritmos de Optimización.Universidad Internacional de Valencia(VIU), Plataforma Virtual.


Python para IA:
Franklin Alvarez Cardinale. (2026). Material y apuntes de la asignatura de Python para la Inteligencia Artificial. Universidad Internacional de Valencia(VIU), Plataforma Virtual.

# Describe brevemente las lineas de como crees que es posible avanzar en el estudio del problema. Ten en cuenta incluso posibles variaciones del problema y/o variaciones al alza del tamaño


El algoritmo desarrollado resuelve con eficiencia el problema para los datos del enunciado, apoyándose en técnicas de memoización para mitigar la explosión combinatoria. Sin embargo, dada la complejidad factorial al escalar el problema, se proponen las siguientes caminos posbles para avanzar en el estudio:

### 1. Optimización del espacio de estados

* **Paralelización del *backtracking*:** Dado que las ramas del árbol de búsqueda son independientes, es posible distribuir la carga computacional entre múltiples núcleos. El desafío técnico residiría en la gestión concurrente de la caché de estados (memoización) para evitar redundancias.
* **Refinamiento de la poda (*Branch and Bound*):** Más allá de las restricciones aritméticas actuales, se podría implementar una estrategia de poda basada en cotas superiores e inferiores (heurísticas) que descarte subárboles completos que, matemáticamente, no puedan alcanzar el valor objetivo, reduciendo así la exploración innecesaria.
* **Optimización de estados:** Si el problema escalara a un número muy elevado de operadores o dígitos, se podría estudiar la transición hacia representaciones de estado mediante máscaras de bits (*bitmasking*). Esto permitiría compactar los estados y detectar equivalencias entre subproblemas de forma más eficiente.

### 2. Variaciones estructurales del problema

* **Incorporación de paréntesis (Árboles de expresión):** Actualmente, el modelo sigue una estructura lineal. Permitir el uso de paréntesis transformaría el problema en la generación de árboles binarios de expresión. Esto aumentaría el espacio de soluciones exponencialmente (modelable mediante los números de Catalan), acercando el estudio a problemas clásicos de generación de expresiones simbólicas.
* **Relajación de restricciones:** Eliminar la restricción de "sin repetición" o añadir operadores complejos (como potenciación o raíces) convertiría el problema en un espacio de estados teóricamente infinito. Este escenario es ideal para evaluar la robustez y convergencia de las heurísticas de poda propuestas.
* **Reorientación del objetivo:** En lugar de realizar una enumeración exhaustiva, el problema podría reformularse como una **tarea de optimización** (ej. encontrar la expresión más cercana a un valor objetivo). Este cambio permitiría aplicar algoritmos de búsqueda heurística (como A* o *Greedy*) o algoritmos evolutivos, orientando el enfoque del estudio hacia la búsqueda de soluciones aproximadas en tiempos computacionales reducidos.